# OMM 데이터 분석 — 4팀 (⑦ 재료 네트워크 · ⑧ 계량 신뢰도)

**사용법**
1. 맨 위 `[0단계] 준비` 셀부터 차례로 실행 (Shift+Enter)
2. 분석 코드를 고치려면 `%%writefile` 셀 안에서 고친 뒤 그 셀을 실행 → `report/analyses/` 의 내 파일이 저장돼요
3. 바로 아래 `[실행]` 셀을 실행하면 고친 코드로 그림이 다시 그려지고, `[데이터 확인]` 셀에서 불러온 DataFrame 을 볼 수 있어요
4. 사이트 반영: 관리자 화면 › 보고서 차트 › **다시 그리기** (또는 매일 18:30 자동). 데이터 분석 탭의 코드 칸은 저장하는 즉시 바뀌어요

> 다른 팀 파일(`analyses/` 의 다른 번호)과 `common.py` 는 고치지 마세요. 공통 부분을 바꿔야 하면 팀에 먼저 알려 주세요.

In [ ]:
# [0단계] 준비 — 공통 코드(common.py) 불러오기 · DB 접속 정보
import os, sys, importlib
REPORT = os.getenv('OMM_REPORT', '/opt/pipeline/api/report')   # back1 기준 분석 코드 폴더
sys.path.insert(0, REPORT)

# DB 접속: API 와 같은 읽기 전용 계정. 비밀번호는 화면에 출력하지 않아요
if not os.getenv('PGPASSWORD'):
    try:   # 설치 때 만든 읽기 전용 DB 접속 파일 (팀원 계정만 읽을 수 있음)
        for line in open(os.path.join(REPORT, '.pgenv'), encoding='utf-8'):
            k, _, v = line.strip().partition('=')
            if k.startswith('PG') and v: os.environ.setdefault(k, v.strip('"\''))
    except (FileNotFoundError, PermissionError):
        from getpass import getpass
        os.environ['PGPASSWORD'] = getpass('DB 비밀번호 (omm_api): ')

# 노트북에서 그린 그림은 notebooks/_out 에 저장 (사이트 그림은 건드리지 않음)
os.environ['REPORT_DIR'] = os.path.join(REPORT, 'notebooks', '_out')
os.makedirs(os.environ['REPORT_DIR'], exist_ok=True)

import common
common.SHOW = True          # save() 할 때 그림을 셀 아래에 바로 보여 주기
from common import *
print('준비 완료')

## ⑦ 재료 네트워크
레시피 간 식재료 공유 네트워크 → 레시피마다 재료의 연관성

- 파일: `report/analyses/a7_network.py` · 함수: `chart7()`
- 그림 파일: `notebooks/_out/` 에 PNG · SVG 로 저장

In [ ]:
%%writefile ../analyses/a7_network.py
"""⑦ 재료 네트워크 — 레시피 간 식재료 공유 네트워크 → 레시피마다 재료의 연관성
담당: 4팀 (⑦⑧) · 이 파일만 고치면 돼요.
  - 노트북 notebooks/4팀_⑦⑧.ipynb 에서 고치고 바로 실행해 볼 수 있어요.
  - 서버는 매일 18:30 charts_v4.py 가 이 파일의 chart7() 을 불러 그림을 다시 그려요.
  - 관리자 화면 › 데이터 분석 탭의 '차트 그리기' 코드 칸에 이 파일 내용이 그대로 보여요.
"""
from common import *

# [1단계] DB에서 데이터 불러오기 (q = SQL → pandas DataFrame)
# [2단계] pandas 로 가공
# [3단계] matplotlib 으로 그리고 save() 로 저장 (PNG + SVG)

def net_data(n_menu=12, n_ing=14, min_share=0.4):
    d = q("""SELECT m.menu_id, m.name AS menu, m.recipe_count AS mc, i.name AS ing, count(DISTINCT r.recipe_id) AS n
             FROM omm_menus m JOIN omm_recipes r USING (menu_id) JOIN omm_recipe_ingredients ri USING (recipe_id)
             JOIN omm_ingredients i ON i.ingredient_id = ri.ingredient_id AND NOT i.is_seasoning
             GROUP BY m.menu_id, m.name, m.recipe_count""".replace('GROUP BY m.menu_id, m.name, m.recipe_count', 'GROUP BY m.menu_id, m.name, m.recipe_count, i.name'))
    if d.empty: return None
    d['share'] = d.n / d.mc
    omm_menus = d.drop_duplicates('menu').nlargest(n_menu, 'mc').menu.tolist()
    e = d[d.menu.isin(omm_menus) & (d.share >= min_share)]
    deg = e.groupby('ing').menu.nunique()
    ings = deg[deg >= 2].sort_values(ascending=False).head(n_ing).index.tolist() or deg.sort_values(ascending=False).head(n_ing).index.tolist()
    e = e[e.ing.isin(ings)]
    omm_menus = [m for m in omm_menus if m in set(e.menu)]
    # 선이 덜 겹치게: 연결된 반대편 순서의 평균으로 여러 번 정렬
    for _ in range(8):
        pos = {m: i for i, m in enumerate(omm_menus)}
        ings.sort(key=lambda x: e[e.ing == x].menu.map(pos).mean())
        pos = {g: i for i, g in enumerate(ings)}
        omm_menus.sort(key=lambda x: e[e.menu == x].ing.map(pos).mean())
    return ings, omm_menus, e

def chart7():
    nd = net_data()
    if not nd: return empty('7_네트워크.png', 7, '레시피 간 식재료 공유 네트워크', '메뉴 · 재료 데이터 없음')
    ings, omm_menus, e = nd
    H = max(len(ings), len(omm_menus))
    yi = {g: (i + 0.5) * H / len(ings) for i, g in enumerate(ings)}; ym = {m: (i + 0.5) * H / len(omm_menus) for i, m in enumerate(omm_menus)}
    fig, ax = plt.subplots(figsize=(11, 1.4 + H * 0.95)); ax.axis('off'); ax.grid(False)
    for _, r in e.iterrows():
        ax.plot([0.12, 0.86], [yi[r.ing], ym[r.menu]], color='#9b9a95', lw=0.6 + r.share * 3, alpha=0.35 + r.share * 0.4, solid_capstyle='round', zorder=1)
    for g in ings:
        ax.scatter(0.08, yi[g], s=2200, color='#dbe9fb', edgecolor=S[0], linewidth=2, zorder=3)
        ax.text(0.08, yi[g], g, ha='center', va='center', color='#1d4f9a', weight='bold', fontsize=11, zorder=4)
    for m in omm_menus:
        ax.text(0.92, ym[m], m, ha='center', va='center', color='#0b5d1e', weight='bold', fontsize=12, zorder=4,
                bbox=dict(boxstyle='round,pad=0.6,rounding_size=0.8', fc='#e2f7e6', ec='#1a9a3c', lw=2))
    ax.set_xlim(0, 1); ax.set_ylim(H + 0.5, -0.6)
    ax.text(0.08, -0.45, '식재료', ha='center', color=INK2); ax.text(0.92, -0.45, '메뉴', ha='center', color=INK2)
    ax.set_title('⑦ 레시피 간 식재료 공유 네트워크 — 재료 ↔ 메뉴', loc='left')
    ax.text(0, -0.02, '선 = 그 메뉴 레시피의 40% 이상에 들어가는 재료 · 선이 굵을수록 비율 ↑ · 메뉴 2개 이상에 걸친 재료만 (양념 제외) · 마우스로 보려면 7_네트워크.html',
            transform=ax.transAxes, color=MUTED, fontsize=11)
    save(fig, '7_네트워크.png', 7, '재료 ↔ 메뉴 공유 네트워크', '같은 재료를 쓰는 메뉴끼리 이어짐')
    net_html(ings, omm_menus, e)
    n = q("""SELECT i.name, n.degree, n.betweenness, n.community_id FROM omm_ingredient_network_nodes n JOIN omm_ingredients i USING (ingredient_id)
             ORDER BY n.degree DESC, n.betweenness DESC LIMIT 15""")
    if not n.empty:
        fig, ax = plt.subplots(figsize=(9, 0.6 + len(n) * 0.42)); n = n[::-1]
        ax.barh(n.name, n.degree, color=S[0], height=0.6)
        for y, (dg, b) in enumerate(zip(n.degree, n.betweenness)): ax.text(dg, y, f'  {dg}개 재료와 연결 · 매개 {b:.2f}', va='center', fontsize=11, color=INK2)
        ax.set_title('만능 재료 — 함께 쓰이는 재료 수 (연결 중심성)'); ax.grid(axis='y', visible=False)
        note(ax, '매개 중심성이 높을수록 서로 다른 요리군을 잇는 "다리" 재료')
        save(fig, '7_중심성.png', 7, '만능 재료 순위', '연결 · 매개 중심성')

def net_html(ings, omm_menus, e):
    W, row = 760, 64; H = max(len(ings), len(omm_menus)) * row + 40
    yi = {g: 30 + (i + 0.5) * (H - 40) / len(ings) for i, g in enumerate(ings)}; ym = {m: 30 + (i + 0.5) * (H - 40) / len(omm_menus) for i, m in enumerate(omm_menus)}
    esc = html.escape
    lines = ''.join(f'<line x1="130" y1="{yi[r.ing]:.0f}" x2="560" y2="{ym[r.menu]:.0f}" data-i="{esc(r.ing)}" data-m="{esc(r.menu)}" '
                    f'stroke-width="{1 + r.share * 4:.1f}"><title>{esc(r.ing)} → {esc(r.menu)}: 레시피의 {r.share * 100:.0f}%</title></line>' for _, r in e.iterrows())
    nodes = ''.join(f'<g class="ing" data-i="{esc(g)}"><circle cx="90" cy="{yi[g]:.0f}" r="28"/><text x="90" y="{yi[g]:.0f}">{esc(g)}</text></g>' for g in ings)
    nodes += ''.join(f'<g class="menu" data-m="{esc(m)}"><rect x="560" y="{ym[m] - 22:.0f}" width="170" height="44" rx="12"/><text x="645" y="{ym[m]:.0f}">{esc(m)}</text></g>' for m in omm_menus)
    rows = ''.join(f'<tr><td>{esc(r.ing)}</td><td>{esc(r.menu)}</td><td>{r.share * 100:.0f}%</td></tr>' for _, r in e.sort_values(['menu', 'share'], ascending=[True, False]).iterrows())
    page = f'''<!DOCTYPE html><html lang="ko"><head><meta charset="utf-8"><title>⑦ 재료 ↔ 메뉴 공유 네트워크</title><style>
body{{margin:0;padding:24px;background:#fcfcfb;color:#0b0b0b;font-family:"Malgun Gothic","Apple SD Gothic Neo",sans-serif}}
h1{{font-size:20px;margin:0 0 4px}} p{{color:#52514e;margin:0 0 16px;font-size:14px}}
svg line{{stroke:#9b9a95;stroke-opacity:.45;stroke-linecap:round;transition:.15s}}
svg text{{text-anchor:middle;dominant-baseline:central;font-weight:700;font-size:14px;pointer-events:none}}
.ing circle{{fill:#dbe9fb;stroke:#2a78d6;stroke-width:2}} .ing text{{fill:#1d4f9a}}
.menu rect{{fill:#e2f7e6;stroke:#1a9a3c;stroke-width:2}} .menu text{{fill:#0b5d1e}}
.ing,.menu{{cursor:pointer}} .dim{{opacity:.15}} line.on{{stroke:#2a78d6;stroke-opacity:.9}}
details{{margin-top:16px}} table{{border-collapse:collapse;font-size:13px}} td{{padding:3px 10px;border-bottom:1px solid #e6e5e0}}
</style></head><body><h1>⑦ 레시피 간 식재료 공유 네트워크</h1>
<p>왼쪽 재료나 오른쪽 메뉴에 마우스를 올리면 연결된 것만 강조돼요 · 선 굵기 = 그 메뉴 레시피 중 재료가 들어간 비율 · 선 위에 올리면 비율이 보여요</p>
<svg viewBox="0 0 {W} {H}" width="{W}" height="{H}"><text x="90" y="12" style="font-weight:400;fill:#52514e">식재료</text>
<text x="645" y="12" style="font-weight:400;fill:#52514e">메뉴</text>{lines}{nodes}</svg>
<details><summary>표로 보기</summary><table><tr><td><b>재료</b></td><td><b>메뉴</b></td><td><b>비율</b></td></tr>{rows}</table></details>
<script>
const L=[...document.querySelectorAll('line')],G=[...document.querySelectorAll('.ing,.menu')];
function hi(key,val){{const on=new Set();L.forEach(l=>{{const h=l.dataset[key]===val;l.classList.toggle('on',h);l.classList.toggle('dim',!h);if(h){{on.add('i:'+l.dataset.i);on.add('m:'+l.dataset.m)}}}});
G.forEach(g=>g.classList.toggle('dim',!on.has(g.dataset.i!==undefined?'i:'+g.dataset.i:'m:'+g.dataset.m)))}}
function off(){{L.forEach(l=>l.classList.remove('on','dim'));G.forEach(g=>g.classList.remove('dim'))}}
G.forEach(g=>{{g.addEventListener('mouseenter',()=>g.dataset.i!==undefined?hi('i',g.dataset.i):hi('m',g.dataset.m));g.addEventListener('mouseleave',off)}});
</script></body></html>'''
    open(os.path.join(OUT, '7_네트워크.html'), 'w', encoding='utf-8').write(page); print('  ✔ 7_네트워크.html')

if __name__ == '__main__':
    chart7()

In [ ]:
# [실행] 저장한 파일을 다시 읽어서 그림 그리기
from analyses import a7_network
importlib.reload(a7_network)
common.queries()          # 이전 기록 비우기
a7_network.chart7()

In [ ]:
# [데이터 확인] ⑦ 재료 네트워크 이 DB 에서 불러온 DataFrame (SQL → pandas)
for i, (sql, df) in enumerate(common.queries(), 1):
    print(f'── 불러온 데이터 {i} · {len(df):,}행 × {df.shape[1]}열')
    display(df.head(10))

## ⑧ 계량 신뢰도
"넉넉히 · 약간"처럼 대충 적은 레시피는 신뢰도가 낮음 → 단위로 적힌 레시피와 분류해 차트화

- 파일: `report/analyses/a8_measure.py` · 함수: `chart8()`
- 그림 파일: `notebooks/_out/` 에 PNG · SVG 로 저장

In [ ]:
%%writefile ../analyses/a8_measure.py
"""⑧ 계량 신뢰도 — "넉넉히 · 약간" 같은 모호 표기 비율로 레시피 신뢰도 분류
담당: 4팀 (⑦⑧) · 이 파일만 고치면 돼요.
  - 노트북 notebooks/4팀_⑦⑧.ipynb 에서 고치고 바로 실행해 볼 수 있어요.
  - 서버는 매일 18:30 charts_v4.py 가 이 파일의 chart8() 을 불러 그림을 다시 그려요.
  - 관리자 화면 › 데이터 분석 탭의 '차트 그리기' 코드 칸에 이 파일 내용이 그대로 보여요.
"""
from common import *

# [1단계] DB에서 데이터 불러오기 (q = SQL → pandas DataFrame)
# [2단계] pandas 로 가공
# [3단계] matplotlib 으로 그리고 save() 로 저장 (PNG + SVG)

def chart8():
    a = q("SELECT amount_class AS c, count(*) AS n FROM omm_recipe_ingredients GROUP BY 1")
    if a.empty: return empty('8_신뢰도.png', 8, '수량 표기 신뢰도', '레시피 재료 데이터 없음')
    lab = {'AMOUNT_MEASURED': '정량 (g · T · 컵)', 'AMOUNT_COUNTED': '개수 (개 · 모 · 대)', 'AMOUNT_VAGUE': '모호 (약간 · 넉넉히)', 'AMOUNT_MISSING': '수량 없음'}
    col = {'AMOUNT_MEASURED': S[0], 'AMOUNT_COUNTED': S[2], 'AMOUNT_VAGUE': S[1], 'AMOUNT_MISSING': MUTED}
    a = a.set_index('c').reindex(list(lab)).fillna(0)
    g = q("""SELECT rm.reliability_grade AS g, count(*) AS n, avg(r.avg_rating) AS r, percentile_cont(0.5) WITHIN GROUP (ORDER BY r.view_count) AS v
             FROM omm_recipe_metrics rm JOIN omm_recipes r USING (recipe_id) WHERE rm.reliability_grade IS NOT NULL GROUP BY 1 ORDER BY 1""")
    v = q("SELECT unit AS u, count(*) AS n FROM omm_recipe_ingredients WHERE amount_class = 'AMOUNT_VAGUE' AND unit IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 10")
    fig, axs = plt.subplots(1, 4, figsize=(17, 6), gridspec_kw={'width_ratios': [1.2, 1, 1, 1]})
    w, _ = axs[0].pie(a.n, colors=[col[c] for c in a.index], startangle=90, counterclock=False,
                      wedgeprops={'width': 0.38, 'edgecolor': SURF, 'linewidth': 2})
    tot = a.n.sum()
    axs[0].legend(w, [f'{lab[c]}  {n / tot * 100:.1f}%' for c, n in zip(a.index, a.n)], loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=2, fontsize=11)
    axs[0].text(0, 0, f'재료줄\n{tot:,.0f}', ha='center', va='center', color=INK, weight='bold'); axs[0].set_title('⑧ 재료 수량 표기 방식')
    gc = {'REL_A': S[0], 'REL_B': S[2], 'REL_C': S[3], 'REL_D': S[1]}
    g['lbl'] = g.g.str.replace('REL_', '', regex=False)     # 화면에는 A · B · C · D
    axs[1].bar(g.lbl, g.n, color=[gc[x] for x in g.g], width=0.6)
    for i, n in enumerate(g.n): axs[1].text(i, n, f'{n:,}', ha='center', va='bottom', color=INK2)
    axs[1].set_title('레시피 신뢰도 등급'); axs[1].grid(axis='x', visible=False)
    axs[2].bar(g.lbl, g.r, color=[gc[x] for x in g.g], width=0.6)
    for i, r in enumerate(g.r): axs[2].text(i, r, f'★{r:.2f}', ha='center', va='bottom', color=INK)
    axs[2].set_ylim(max(0, g.r.min() - 0.8) if len(g) else 0, 5.2); axs[2].set_title('등급별 평균 별점'); axs[2].grid(axis='x', visible=False)
    vv = v[::-1]
    axs[3].barh(vv.u, vv.n, color=S[1], height=0.6)
    for y, n in enumerate(vv.n): axs[3].text(n, y, f'  {n:,}', va='center', color=INK2, fontsize=11)
    axs[3].set_title('자주 쓰는 모호한 표현'); axs[3].grid(axis='y', visible=False)
    note(axs[1], '등급 = 정량 · 개수로 적은 재료 비율 · A ≥ 90% · B ≥ 70% · C ≥ 50% · D < 50%')
    save(fig, '8_신뢰도.png', 8, '수량 표기 신뢰도', '"약간 · 넉넉히"처럼 대충 적은 레시피 비율과 별점')

if __name__ == '__main__':
    chart8()

In [ ]:
# [실행] 저장한 파일을 다시 읽어서 그림 그리기
from analyses import a8_measure
importlib.reload(a8_measure)
common.queries()          # 이전 기록 비우기
a8_measure.chart8()

In [ ]:
# [데이터 확인] ⑧ 계량 신뢰도 이 DB 에서 불러온 DataFrame (SQL → pandas)
for i, (sql, df) in enumerate(common.queries(), 1):
    print(f'── 불러온 데이터 {i} · {len(df):,}행 × {df.shape[1]}열')
    display(df.head(10))

---
### 결과 정리 (직접 작성)
- 분석 결과:
- 서비스에 활용한 곳:
- 수익 모델과의 연결: